In [1]:
import requests
from bs4 import BeautifulSoup
from pprint import pprint
import os 

In [2]:
url = 'https://kotlinlang.org/api/core/kotlin-stdlib/kotlin/array-of.html'
r = requests.get(url,) 

In [3]:
import sys
import http.client
import json
import subprocess

def ask_llm(question: str, model: str = "deepseek-r1:1.5b") -> str:
    """
    Send a prompt to local Ollama and return the model's answer as a string.
    Uses only Python's standard library.
    """
    conn = http.client.HTTPConnection("192.168.0.228", 11434)

    payload = json.dumps({"model": model, "prompt": question, "stream": False})

    headers = {"Content-Type": "application/json"}

    conn.request("POST", "/api/generate", body=payload, headers=headers)
    response = conn.getresponse()

    if response.status != 200:
        raise RuntimeError(f"Ollama error: {response.status} {response.reason}")

    data = response.read().decode("utf-8")
    conn.close()

    result = json.loads(data)
    return result.get("response", "")

In [4]:
soup = BeautifulSoup(r.text)

In [5]:
found = soup.find_all("div", {"class":"symbol monospace"})
func_sig = ""
if len(found) > 0: 
    func_sig = found[0].text
func_sig

'expect inline fun <T> arrayOf(vararg elements: T): Array<T>(source)'

In [6]:
found = soup.find_all("p", {"class": "paragraph"})
if len(found) > 0: 
    func_doc = found[0].text
func_doc

'Returns an array containing the specified elements.'

In [7]:
found = soup.find_all("div", {"class":"breadcrumbs"})
func_path = ""
if len(found) > 0: 
    func_path = found[0].text
func_path

'kotlin-stdlib/kotlin/arrayOf'

In [8]:
found = soup.find_all("h1", {"class":"cover"})
func_heading = ""
if len(found) > 0: 
    func_heading = found[0].text
func_heading

'arrayOf'

In [9]:
prompt = f"""
I want you to give me a example usage of the kotlin {func_heading}.

Here is some info about it: 
{func_heading}
{func_path}
{func_doc}
{func_sig}
Source: {url}

The example will be used for documentation. 
It should be easy to follow and understand.

So what would be a simple example usage of {func_sig}
Only give the code example and nothing else in your response.
"""

In [10]:
response = ask_llm(prompt, "gpt-oss")

In [11]:
bredcrum = ""
bpath = func_path.split("/")[:-1]
for i, crum in enumerate(bpath):
    bredcrum += f"/ [{crum}](/{"/".join(bpath[:i+1])}) "
bredcrum += f"/ [{func_heading}]({func_path})"
bredcrum = bredcrum[1:]
bredcrum

' [kotlin-stdlib](/kotlin-stdlib) / [kotlin](/kotlin-stdlib/kotlin) / [arrayOf](kotlin-stdlib/kotlin/arrayOf)'

In [12]:
md_text = f"""

# {func_heading}
{bredcrum}

{func_doc}

```kotlin
{func_sig}
```

```kotlin
{response}
```

[Source]({url})

"""

In [13]:
md_dir = f"./docs/{"/".join(func_path.split("/")[:-1])}" 
md_file = f"{md_dir}/{func_heading}.md"
os.system(f"mkdir -p {md_dir}")
with open(md_file, "w") as f:
    f.write(md_text)


In [21]:
import asyncio
from playwright.async_api import async_playwright

async def get_html(page_url:str):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(page_url)
        await page.wait_for_load_state("networkidle")

        html = await page.content()
        await browser.close()
        return html

In [22]:
soup = BeautifulSoup(await get_html(url))

In [45]:
li = soup.find_all("a", {"class": "toc--link"})
stdlib_urls = [l["href"].replace("../../", "https://kotlinlang.org/api/core/") for l in li if "stdlib" in l["href"] and not "index.html" in l["href"]]

In [46]:
import json
with open('./kotlin_stdlib_urls.json', 'w') as f:
    json.dump(stdlib_urls, f)

In [49]:
def walk_dirs(directory: str, recursive=False, follow_symlinks=False):
    for entry in os.scandir(directory):
        if entry.is_dir(follow_symlinks=follow_symlinks):
            yield entry.path
            if recursive:
                yield from walk_dirs(entry.path, recursive=True, follow_symlinks=follow_symlinks)

paths = [d for d in walk_dirs("./docs/", recursive=True)]
paths


['./docs/kotlin-stdlib', './docs/kotlin-stdlib/kotlin']

In [ ]:
from os import listdir
from os.path import isfile, join

for path in paths: 
    onlyfiles = [f for f in listdir(path) if isfile(join(path, f))]
    for f in onlyfiles:
        print(f"* {f} ") 


| apply.md | |
| addSuppressed.md | |
| arrayOf.md | |
| and.md | |
| also.md | |
